# Session 3 — Two Qubits, One Fate: Entanglement, the 85% Game & the Real Machine
### Quantum Computing Workshop · Session 3 of 4

**The rituals still rule:**
- Run cells **top to bottom**.  **YOUR TURN** = fill in a blank.  **PREDICT FIRST** = type your prediction *before* running.
- Pairs: driver types, navigator reads aloud. **Swap at Task 3.**

> First: **File → Save a copy in Drive**.

## Cell 1 — Setup (run first, ~1 minute)

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib pylatexenc

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator, partial_trace
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import numpy as np

simulator = AerSimulator()

def show_recipe(qc):
    """Print the state's amplitudes — works for 1 OR 2 qubits."""
    sv = Statevector(qc)
    n = sv.num_qubits
    amps = np.round(sv.data, 3)
    for idx, a in enumerate(amps):
        label = format(idx, f'0{n}b')
        print(f"amplitude of |{label}⟩:  {a}")

def show_ball(qc):
    """Draw a Bloch sphere for EACH qubit in a (measurement-free) circuit."""
    return plot_bloch_multivector(Statevector(qc))

def run_counts(qc, shots=1000):
    return simulator.run(qc, shots=shots).result().get_counts()

print("✅ Setup complete — bring on the second qubit.")

---
## Task 1 — Build the strangest state in physics

Two lines. That's all a Bell pair costs:

```
qc.h(0)       # qubit 0 becomes the spinning coin |+⟩
qc.cx(0, 1)   # CNOT: qubit 0 controls, qubit 1 is the target
```

**PREDICT FIRST:** the recipe starts as `(|00⟩ + |01⟩)/√2` after the `h`... wait, which qubit is which digit? In these printouts the label reads **qubit 1, then qubit 0** — so after `h` on qubit 0 the live terms are |00⟩ and |01⟩. Now apply the CNOT rule (*if the control's part is 1, flip the target's part*) to each term. What four amplitudes should print?

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
show_recipe(bell)   # prediction: ______

Two amplitudes of 0.707, two of exactly zero — the |01⟩ and |10⟩ parts are simply **absent**. Now let 1,000 measurements confirm what the recipe promises:

In [ ]:
bell_m = QuantumCircuit(2, 2)
bell_m.h(0)
bell_m.cx(0, 1)
bell_m.measure([0, 1], [0, 1])
counts = run_counts(bell_m)
print(counts)
plot_histogram(counts)

Only `00` and `11`. Two coins, each individually 50/50 random — and yet they **always agree**. Nobody sent a signal. The agreement was baked into one shared recipe.

**YOUR TURN:** build the *always-disagree* pair. **PREDICT FIRST:** which single gate, added to the Bell recipe, converts {00, 11} into {01, 10}? Build it, print the recipe, run the counts.

In [ ]:
# your always-disagree pair here


<details><summary> Solution</summary>

Add `qc.x(1)` (or `qc.x(0)`) after the CNOT — flipping either qubit's part turns every |00⟩ into |01⟩ and every |11⟩ into |10⟩:

```python
qc = QuantumCircuit(2, 2)
qc.h(0); qc.cx(0, 1); qc.x(1)
qc.measure([0, 1], [0, 1])
print(run_counts(qc))   # only 01 and 10
```
Perfectly anti-correlated — still entangled, still spooky, just contrary. (This is a second member of the four-state **Bell family** — Explorer Zone E2 collects the whole set.)
</details>

---
## Task 2 — Watch the globes vanish 

Session 2's big claim: *every* one-qubit state is a point on the Bloch sphere. This morning's bigger claim: entangled qubits **have no individual points**.

First, a control experiment — two qubits that are each in superposition but NOT entangled:

In [ ]:
product = QuantumCircuit(2)
product.h(0)
product.h(1)
show_ball(product)   # two arrows, both pointing east at |+⟩ — each qubit has its own address

**PREDICT FIRST:** now draw the Bell pair's spheres. Where do the two arrows point?

In [ ]:
show_ball(bell)   # prediction: ______

**No arrows.** Both shrink to a dot at the center of the ball — the mathematical way of saying *this qubit, considered alone, points nowhere at all*. All the definiteness lives in the **pair**; the parts have gone individually blank.

That's Theory C, rendered: entangled qubits don't share a globe — they give up their globes.

<details><summary> Why the center means "no address"</summary>

Each qubit alone behaves like a 50/50 coin **no matter which axis you measure along** — pole question, east-west question, any tilt: pure noise. The only point equally far from every answer is the center. (A lone qubit at the center is called *maximally mixed* — maximum ignorance.)
</details>

---
## Task 3 — Play the 75% game at 85% 
*(Drivers and navigators: swap seats!)*

Your pennies could not beat 75%. The theorem says no pre-agreed plan ever will. So how do entangled players break through? Here is the whole idea, built from things you already know:

**The strategy.** Alice and Bob each carry one qubit of a shared Bell pair `(|00⟩+|11⟩)/√2`. Instead of answering from a memorized table, each one **measures their qubit** — and their question decides *which axis they measure along* (they tilt the axis on the Bloch sphere). The measurement outcome (0 or 1) **is** their answer.

**Why tilting is the trick.** Recall the key fact about a Bell pair from Task 1: measure both qubits *the same way* and they **always agree**. Now the deeper fact — for two measurement axes an angle θ apart, the two outcomes agree with probability **cos²(θ/2)**:
- **0° apart** (same axis) → agree 100% ✓ (the Task 1 result)
- **45° apart** → agree ~85%
- **135° apart** → agree only ~15%, i.e. they **disagree** ~85%

**The clever wiring.** Alice measures along `0°` or `90°`; Bob along `45°` or `−45°`. Work out the gap for each of the four question-pairs:

| Alice question | Bob question | axis gap | outcome | rule wants |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | 45° | agree ~85% | agree ✓ |
| 0 | 1 | 45° | agree ~85% | agree ✓ |
| 1 | 0 | 45° | agree ~85% | agree ✓ |
| 1 | 1 | **135°** | **disagree ~85%** | **disagree ✓** |

In every row, the physics does what the win-rule asks — about 85% of the time. The `(1,1)` round is special in the game (you must disagree) *and* special in the geometry (the axes land far apart). That alignment is the entire magic. No fixed answer sheet can imitate it, because a sheet is written before the questions arrive — it can't re-aim an axis on the fly. Entanglement can.

Below, watch the simulator play it out.

In [ ]:
def chsh_game(alice_tilts, bob_tilts, rounds=1000):
    """Play CHSH with tilted measurements. tilts = [angle for question 0, angle for question 1] (radians)."""
    rng = np.random.default_rng()
    wins = 0
    # pre-build the four circuits (one per question combo)
    circuits = {}
    for x in [0, 1]:
        for y in [0, 1]:
            qc = QuantumCircuit(2, 2)
            qc.h(0); qc.cx(0, 1)              # the shared Bell pair
            qc.ry(-alice_tilts[x], 0)         # Alice tilts her axis by her question's angle
            qc.ry(-bob_tilts[y], 1)           # Bob tilts his
            qc.measure([0, 1], [0, 1])
            circuits[(x, y)] = qc
    for _ in range(rounds):
        x, y = rng.integers(0, 2), rng.integers(0, 2)
        bits = list(run_counts(circuits[(x, y)], shots=1).keys())[0]
        b, a = int(bits[0]), int(bits[1])     # qiskit prints qubit 1's bit first
        if (a ^ b) == (x & y):
            wins += 1
    return wins / rounds

# First, sanity-check a CLASSICAL plan: ignore the qubits, both always answer 0
# (tilt 0 for every question = both measure the pole axis = perfectly agreeing answers)
print("always-agree plan:", chsh_game([0, 0], [0, 0], rounds=400))

~75% — the wall, reproduced in code.

**PREDICT FIRST:** now the quantum plan — Alice tilts `[0, π/2]`, Bob tilts `[π/4, −π/4]`. That's exactly the axis arrangement from the table above: three question-pairs land 45° apart, and the `(1,1)` pair lands 135° apart. The overall win rate works out to **cos²(22.5°)** (the 22.5° is the half-angle in `cos²(θ/2)` with θ = 45°). Above or below your pennies?

In [ ]:
score = chsh_game([0, np.pi/2], [np.pi/4, -np.pi/4], rounds=1000)
print(f"entangled strategy: {score:.1%}   (theory: {np.cos(np.pi/8)**2:.1%})")

**Through the wall.** No communication, no pre-agreed answer sheet — a shared quantum state and well-chosen questions.

 **YOUR TURN:** is 22.5° special, or would any tilt do? Keep Alice at `[0, π/2]` and try Bob at `[π/3, −π/3]`, then `[π/6, −π/6]`, then anything you like. Can you find tilts that score even *higher* than π/4 spacing?

<details><summary> What you'll find</summary>

Every other spacing scores lower — π/4 (giving 22.5° gaps between the four axes) is the proven optimum, and cos²(π/8) ≈ 85.36% is the ceiling for quantum players too (Tsirelson's bound). Even entanglement has a wall — just a higher one. Exploration E3 turns this into a proper angle-sweep experiment with a plot.
</details>

---
## Submitting to the real machine
*(Matt will run this on the projector, live, before the BB84 block. Students: skip ahead to Task 4 — you'll retrieve what this cell submits.)*

In [ ]:
# Matt ONLY — requires an IBM Quantum account (see the Game Kit pipeline guide)
# %pip install -q qiskit-ibm-runtime
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler import generate_preset_pass_manager
# #
# # # One-time on this runtime (paste your API key from quantum.cloud.ibm.com):
# QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="token", overwrite=True)
# #
# service = QiskitRuntimeService(channel="ibm_quantum_platform")
# backend = service.least_busy(operational=True, simulator=False)   # or: service.backend("ibm_kingston")
# print("machine:", backend.name, "|", backend.num_qubits, "qubits")
# #
# qc = QuantumCircuit(2)
# qc.h(0); qc.cx(0, 1)
# qc.measure_all()
# #
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
# isa = pm.run(qc)
# print("compiled gate counts:", dict(isa.count_ops()))   # golf, professionally
# #
# sampler = Sampler(mode=backend)
# job = sampler.run([isa], shots=4096)
# print("JOB ID →", job.job_id())   # write this on the whiteboard!

---
## Task 4 — Collect your results from the real machine 

Somewhere in a lab, a chip at 0.015 kelvin ran your class's Bell pair. Paste the **job ID from the whiteboard** below.

*(Queue still busy? Totally normal — shared hardware has lines. Run the honest-fallback cell two below instead, and retrieve the real thing next session.)*

In [ ]:
%pip install -q qiskit-ibm-runtime
from qiskit_ibm_runtime import QiskitRuntimeService

# One-time: paste the class API key if your facilitator provides one
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="token", overwrite=True)

service = QiskitRuntimeService(channel="ibm_quantum_platform")
job = service.job("job_id")
print("status:", job.status())

result = job.result()
hw_counts = result[0].data.meas.get_counts()
print(hw_counts)
plot_histogram(hw_counts)

**Honest fallback** (clearly labeled: this is a *simulation with a realistic noise model*, standing in until the queue clears):

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.004, 1), ['h', 'ry', 'x'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ['cx'])
nm.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.03, 0.97]]))

hw_counts = AerSimulator(noise_model=nm).run(bell_m, shots=4096).result().get_counts()
print("NOISY-SIM stand-in:", hw_counts)
plot_histogram(hw_counts)

 **YOUR TURN — read the noise like a professional.** The simulator said 01 and 10 are *impossible*. Reality disagrees, slightly. Compute your error rate:

In [ ]:
total = sum(hw_counts.values())
impossible = hw_counts.get('01', 0) + hw_counts.get('10', 0)
print(f"'impossible' outcomes: {impossible} of {total}  →  error rate {impossible/total:.1%}")

<details><summary> What that number means</summary>

Those are decoherence and readout slips — stray heat and imperfect microwave pulses nudging the fragile qubits (slide 4's warning, quantified). A few percent is typical and *good* for today's hardware. Whole teams exist to measure, model, and shrink this number — error-mitigation and calibration engineering are among the most-hired quantum specialties. You just did the first step of their job: quantify the gap between math and machine.
</details>

---
## Task 5 — Capstone: teleport a quantum state 

The setup: qubit 0 holds a **secret state** — a dial setting only you choose. Qubits 1 and 2 are a Bell pair: qubit 1 stays with the sender, qubit 2 belongs to the receiver. The protocol moves the secret from qubit 0 to qubit 2 **without qubit 0's state ever crossing the wire**.

Set your secret dial:

In [ ]:
secret_angle = 1.047   # ← change me! any radians you like

# First: what does the secret look like? (so we can verify delivery later)
peek = QuantumCircuit(1)
peek.ry(secret_angle, 0)
show_ball(peek)

**PREDICT FIRST:** after the protocol runs, three spheres will print. Which qubit's arrow should match the picture above — and what do you expect the other two to look like?

In [ ]:
tp = QuantumCircuit(3)
tp.ry(secret_angle, 0)   # the secret, on qubit 0
tp.h(1)                  # ┐
tp.cx(1, 2)              # ┘ Bell pair shared between qubit 1 (sender) and qubit 2 (receiver)
tp.cx(0, 1)              # ┐ the sender entangles the secret
tp.h(0)                  # ┘ with their half of the pair
tp.cx(1, 2)              # ┐ the receiver's two corrections
tp.cz(0, 2)              # ┘ (in the classic version these are chosen by 2 ordinary bits Alice sends)
show_ball(tp)

Qubit 2's arrow sits **exactly** on your secret dial — and qubits 0 and 1 have been reset to ordinary states with no memory of the secret. Moved, not copied.

**YOUR TURN:** change `secret_angle` to something only you know, re-run both cells, and confirm delivery. Then try a secret with *longitude*: add `peek.rz(0.8, 0)` / `tp.rz(0.8, 0)` right after each `ry` — does phase teleport too?

<details><summary> What just happened (and the fine print)</summary>

Phase teleports too — the *entire* state moves, dial and longitude and all. Fine print worth owning: (1) the Bell pair is **consumed** — one teleport per pair; (2) in the standard protocol Alice *measures* her two qubits and texts Bob 2 ordinary bits telling him which corrections to apply — so nothing outruns light; our version applies the corrections as quantum-controlled gates, which is equivalent and lets you SEE the result without measuring; (3) the original is unavoidably reset — the same no-copying law that stops Eve in BB84. Quantum states move; they never photocopy.
</details>

---
---
#  Exploration (optional, any order — predict before every run!)

### Task E1 — Three coins, one fate
Extend the Bell recipe to 3 qubits: `h(0)`, then `cx(0,1)`, then `cx(1,2)`. **Predict the counts first.**

In [ ]:
# build the 3-qubit version here (use QuantumCircuit(3, 3) and measure all three)

<details><summary> Solution</summary>

```python
ghz = QuantumCircuit(3, 3)
ghz.h(0); ghz.cx(0, 1); ghz.cx(1, 2)
ghz.measure([0,1,2], [0,1,2])
print(run_counts(ghz))   # only 000 and 111
```
Three coins in perfect lockstep — a **GHZ state**. Entanglement isn't a two-qubit trick; it scales, and big entangled states are exactly what quantum computers compute with.
</details>

### Task E2 — The Bell family portrait
There are exactly four Bell states: {00,11} agreeing with a + sign, the same with a − sign, and the two {01,10} versions. You built two already. **Build all four** and print each recipe. (Hint: your Session 2 toolbox — which gate places a minus sign on the |1⟩-part?)

In [ ]:
# the four Bell states here


<details><summary> Solution</summary>

```python
def bell_state(sign_flip=False, anti=False):
    qc = QuantumCircuit(2)
    qc.h(0); qc.cx(0, 1)
    if sign_flip: qc.z(0)    # places the − sign:  (|00⟩ − |11⟩)/√2
    if anti:      qc.x(1)    # makes it disagree:  |01⟩/|10⟩ family
    return qc

for sf in [False, True]:
    for an in [False, True]:
        print(f"sign_flip={sf}, anti={an}:"); show_recipe(bell_state(sf, an)); print()
```
Four states, mutually distinguishable, each maximally entangled — the alphabet of quantum communication (teleportation and E5's superdense coding both spell with it).
</details>

### Task E3 — The angle-sweep experiment
Turn Task 3's YOUR TURN into real science: sweep Bob's tilt spacing from 0 to π/2 in ~10 steps, record `chsh_game` scores (use `rounds=4000` for speed), and plot score vs angle with matplotlib. Where's the peak? What's the shape?

In [ ]:
# your sweep + plot here


<details><summary> Solution sketch</summary>

```python
import matplotlib.pyplot as plt
spacings = np.linspace(0.01, np.pi/2, 10)
scores = [chsh_game([0, np.pi/2], [t, -t], rounds=400) for t in spacings]
plt.plot(spacings, scores, 'o-'); plt.axhline(0.75, ls='--')
plt.xlabel('Bob tilt (rad)'); plt.ylabel('win rate'); plt.show()
```
Smooth curve, peak at π/4, matching cos²-shaped theory — and every point above the dashed line is a little Nobel-worthy anomaly. Congratulations: you've run a Bell-test parameter scan, which is a real thing graduate students do for years.
</details>

### Task E4 — Code your own Eve
Simulate BB84 with an intercept-and-resend Eve: random bits/bases for Alice, Eve measures in random bases and re-prepares, Bob measures in random bases, sift, compute the disagreement rate in the sifted key. Does it match your card game's ~25%?

In [ ]:
# your BB84 + Eve simulation here (pure Python/numpy is fine — no circuits needed)


<details><summary> Solution</summary>

```python
rng = np.random.default_rng()
N = 20000
a_bit, a_bas = rng.integers(0,2,N), rng.integers(0,2,N)
e_bas, b_bas = rng.integers(0,2,N), rng.integers(0,2,N)
e_bit = np.where(e_bas == a_bas, a_bit, rng.integers(0,2,N))     # wrong basis → fresh coin
b_bit = np.where(b_bas == e_bas, e_bit, rng.integers(0,2,N))     # Bob reads Eve's re-prepared qubit
sift = a_bas == b_bas
print(f"sifted disagreement: {np.mean(a_bit[sift] != b_bit[sift]):.1%}")   # ≈ 25%
```
Quarter of the sifted key disagrees — Eve cannot avoid it, because she cannot know the right basis. The alarm is arithmetic.
</details>

### Task E5 — For experienced coders, Superdense coding
Teleportation spends 1 Bell pair + 2 classical bits to move 1 qubit. The mirror-image protocol spends 1 Bell pair + **1 qubit** to move **2 classical bits**. Scheme: Alice and Bob share a Bell pair; Alice applies one of {nothing, X, Z, ZX} to *her qubit alone* to encode 2 bits; she sends that one qubit to Bob; Bob undoes the Bell recipe (`cx(0,1)` then `h(0)`) and measures both. Implement it and verify all four messages arrive intact.

In [ ]:
def send_two_bits(bit1, bit0):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)       # the shared pair
    # Alice encodes on qubit 0 only:
    # your code here
    # Bob decodes:
    qc.cx(0, 1); qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc


<details><summary> Solution</summary>

```python
def send_two_bits(bit1, bit0):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)
    if bit0: qc.x(0)
    if bit1: qc.z(0)
    qc.cx(0, 1); qc.h(0)
    qc.measure([0, 1], [1, 0])   # this mapping makes the printout read in (bit1, bit0) order
    return qc

for b1 in [0, 1]:
    for b0 in [0, 1]:
        print(f"sent {b1}{b0} → received", run_counts(send_two_bits(b1, b0), shots=100))
```
Each message arrives with 100% certainty — two bits carried by touching only one qubit, because the shared entanglement did half the work in advance. Entanglement is a *resource*, spent like fuel — the accounting mindset behind all of quantum networking.
</details>

---
##  Done!
Today's collection: a Bell pair whose parts have no individual addresses, a game score no written plan can reach, a spy alarm powered by collapse, a histogram from real hardware **with its noise quantified**, and one successfully teleported secret. Next session: the attack the encryption clock warned you about — and the people being hired to survive it. 